# Hafta 4 — Makine Öğrenmesinin Temel Kavramları

Bu defterde model kurmuyoruz; **doğru değerlendirme alışkanlıklarını** kuruyoruz: veri ayrımı, taban modeller, hata ölçütleri, aşırı öğrenme deneyi, çapraz doğrulama ve veri kaçağı.

Veri: `elektrik_tuketimi.csv` (aynı klasörde olmalı — sol panelden yükleyin).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("elektrik_tuketimi.csv")
df = df[df.tuketim_kW < 300].copy()                  # 3. haftadaki temizlik
df["sicaklik_C"] = df["sicaklik_C"].interpolate()
df["klima"] = np.clip(df["sicaklik_C"] - 24, 0, None)   # 3. haftada fark ettiğimiz kırık çizgi -> yeni özellik
df["t"] = df.gun * 24 + df.saat                          # zaman indeksi (saat cinsinden)
print(df.shape); df.head()

## 1. Veri ayrımı: zamana göre mi, rastgele mi?

Saatlik yük bir **zaman serisidir**. Doğru ayrım: geçmişle eğit, gelecekte test et.

In [ ]:
egitim = df[df.gun <= 24]; test = df[df.gun > 24]
print("Eğitim:", len(egitim), "saat  Test:", len(test), "saat")

# Karşılaştırma için rastgele ayrım (YANLIŞ olduğunu göstereceğiz)
from sklearn.model_selection import train_test_split
eg_r, te_r = train_test_split(df, test_size=0.2, random_state=0)
print("Rastgele: eğitim", len(eg_r), "test", len(te_r))

## 2. Hata ölçütlerini kendimiz yazalım (Örnek 4.1)

In [ ]:
def olcutler(y, yp):
    y, yp = np.asarray(y, float), np.asarray(yp, float)
    e = y - yp
    return {"MAE": np.abs(e).mean(), "RMSE": np.sqrt((e**2).mean()),
            "MAPE%": 100*np.mean(np.abs(e)/np.abs(y)), "R2": 1 - (e**2).sum()/((y-y.mean())**2).sum()}

y  = [100, 120, 115, 140, 160]; yp = [105, 115, 125, 135, 150]
print({k: round(v, 3) for k, v in olcutler(y, yp).items()})
# sklearn ile doğrulama
print(mean_absolute_error(y, yp), round(np.sqrt(mean_squared_error(y, yp)), 3), round(r2_score(y, yp), 3))

## 3. Taban modeller

Üç taban: eğitim ortalaması, saatlik profil, saat × gün-tipi profili. Bunları yenmeyen model işe yaramaz.

In [ ]:
def rapor(ad, y, yp):
    m = olcutler(y, yp)
    print(f"{ad:26s} MAE={m['MAE']:6.2f}  RMSE={m['RMSE']:6.2f}  MAPE={m['MAPE%']:5.1f}%  R2={m['R2']:6.3f}")

rapor("Taban: ortalama", test.tuketim_kW, np.full(len(test), egitim.tuketim_kW.mean()))
profil = egitim.groupby("saat")["tuketim_kW"].mean()
rapor("Taban: saatlik profil", test.tuketim_kW, test.saat.map(profil))
profil2 = egitim.groupby(["hafta_sonu", "saat"])["tuketim_kW"].mean()
tahmin2 = [profil2[(h, s)] for h, s in zip(test.hafta_sonu, test.saat)]
rapor("Taban: saat x gün tipi", test.tuketim_kW, tahmin2)

In [ ]:
plt.figure(figsize=(10, 3.5))
seg = test.iloc[:72]
plt.plot(range(72), seg.tuketim_kW.values, "k", label="gerçek")
plt.plot(range(72), seg.saat.map(profil).values, label="saatlik profil")
plt.plot(range(72), tahmin2[:72], label="saat x gün tipi")
plt.xlabel("Test saati"); plt.ylabel("kW"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 4. Rastgele ayrım neden yanıltıcı? (kaçak deneyi)

Zaman indeksi (gün·24 + saat) üzerinde çalışan bir k-en yakın komşu modeli düşünün: rastgele ayrımda bir test saatinin **komşu saatleri** eğitimdedir ve model onları kopyalar. Zamana göre ayrımda ise test saatlerinin komşusu yoktur.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor(n_neighbors=2)
# (a) rastgele ayrım
knn.fit(eg_r[["t"]], eg_r.tuketim_kW); rapor("k-NN(t), rastgele ayrım", te_r.tuketim_kW, knn.predict(te_r[["t"]]))
# (b) zamana göre ayrım
knn.fit(egitim[["t"]], egitim.tuketim_kW); rapor("k-NN(t), zamana göre ayrım", test.tuketim_kW, knn.predict(test[["t"]]))

**Yorum:** Rastgele ayrımda model 'komşu saati kopyalayarak' çok iyi görünür (R² yüksek); zamana göre ayrımda aynı model tamamen çöker, çünkü gelecek için komşusu yoktur. Rastgele ayrımdaki başarı sahada asla gerçekleşmez — bu bir **zaman kaçağıdır**. Genel kural: zaman serisinde zamana göre ayır.

## 5. Aşırı öğrenme deneyi: polinom derecesi

Sıcaklık → tüketim ilişkisine 1'den 12'ye polinomlar uyduralım; eğitim ve doğrulama RMSE'sini derece ile çizelim.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

Xe, ye = egitim[["sicaklik_C"]].values, egitim.tuketim_kW.values
Xt, yt = test[["sicaklik_C"]].values, test.tuketim_kW.values
dereceler = range(1, 13); e_tr, e_te = [], []
for d in dereceler:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(Xe, ye)
    e_tr.append(olcutler(ye, m.predict(Xe))["RMSE"]); e_te.append(olcutler(yt, m.predict(Xt))["RMSE"])

plt.figure(figsize=(7, 3.5))
plt.plot(dereceler, e_tr, "o-", label="eğitim RMSE"); plt.plot(dereceler, e_te, "s-", label="test RMSE")
plt.xlabel("Polinom derecesi"); plt.ylabel("RMSE (kW)"); plt.legend(); plt.grid(alpha=.3); plt.show()
print("En iyi derece (test):", list(dereceler)[int(np.argmin(e_te))])

In [ ]:
# Uydurulan eğrileri görelim
xx = np.linspace(df.sicaklik_C.min(), df.sicaklik_C.max(), 200)[:, None]
plt.figure(figsize=(8, 3.5)); plt.scatter(Xe, ye, s=5, color="gray", label="eğitim")
for d in (1, 3, 12):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(Xe, ye)
    plt.plot(xx, m.predict(xx), lw=2, label=f"derece {d}")
plt.ylim(0, 300); plt.xlabel("°C"); plt.ylabel("kW"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. k-katlı çapraz doğrulama

Zaman serisi için `TimeSeriesSplit` (ileriye dönük): her katta geçmişle eğit, sonraki dilimde doğrula.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, KFold
X = df[["saat", "hafta_sonu", "sicaklik_C", "klima"]].values; yv = df.tuketim_kW.values
tss = TimeSeriesSplit(n_splits=5)
sonuc = []
for k, (i_tr, i_va) in enumerate(tss.split(X)):
    m = LinearRegression().fit(X[i_tr], yv[i_tr])
    r = olcutler(yv[i_va], m.predict(X[i_va]))["RMSE"]; sonuc.append(r)
    print(f"Kat {k+1}: eğitim {len(i_tr):3d} örnek, doğrulama {len(i_va):3d} örnek, RMSE = {r:.2f}")
print(f"RMSE = {np.mean(sonuc):.2f} ± {np.std(sonuc):.2f} kW")

## 7. Veri kaçağı: ölçekleyiciyi doğru kullanmak (Örnek 4.4)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc_dogru = StandardScaler().fit(egitim[["sicaklik_C"]])     # sadece eğitim
sc_yanlis = StandardScaler().fit(df[["sicaklik_C"]])         # tüm veri (KAÇAK)
ornek = pd.DataFrame({"sicaklik_C": [28.0]})
print("Eğitim μ, σ:", sc_dogru.mean_[0].round(2), np.sqrt(sc_dogru.var_[0]).round(2), " -> z =", sc_dogru.transform(ornek)[0, 0].round(3))
print("Tüm veri μ, σ:", sc_yanlis.mean_[0].round(2), np.sqrt(sc_yanlis.var_[0]).round(2), " -> z =", sc_yanlis.transform(ornek)[0, 0].round(3))

Bu veri setinde eğitim ve tüm-veri parametreleri birbirine yakın çıkar (son günler belirgin şekilde daha sıcak değil); Örnek 4.4'teki sayılar mekanizmayı abartılı göstermek için seçilmişti. Önemli olan sıra: **fit sadece eğitimde**.

**Doğru kalıp — Pipeline:** ölçekleyici ve model tek nesnede; `fit` sadece eğitim verisine uygulanır, test otomatik olarak yalnızca `transform` görür.

In [ ]:
pipe = make_pipeline(StandardScaler(), LinearRegression())
pipe.fit(egitim[["saat", "hafta_sonu", "sicaklik_C", "klima"]], egitim.tuketim_kW)
rapor("Pipeline (doğrusal, 4 özellik)", test.tuketim_kW, pipe.predict(test[["saat", "hafta_sonu", "sicaklik_C", "klima"]]))

**Dikkat:** Bu doğrusal model 'saat × gün tipi' taban modelini yenemedi (R² 0.54 < 0.92). Sebep: doğrusal model saati düz bir çizgi sanıyor, oysa günlük profil tepeli. 5. haftada saat için daha iyi özellikler (sin/cos kodlama, one-hot) ile tabanı geçeceğiz — özellik mühendisliğinin gücü.

## 8. Alıştırmalar

**Alıştırma 1.** `olcutler` fonksiyonuna MSE'yi de ekleyin ve y = [50, 200], ŷ = [60, 190] için tüm ölçütleri hesaplayın. MAPE'nin neden ilk örneği daha çok cezalandırdığını bir cümleyle açıklayın.

In [ ]:
# Alıştırma 1

**Alıştırma 2.** 'Dünün aynı saati' (naif tahmin) taban modelini kurun: test kümesindeki her saat için 24 saat öncesinin gerçek tüketimini tahmin olarak kullanın. RMSE'sini diğer tabanlarla karşılaştırın. (İpucu: `df.tuketim_kW.shift(24)`)

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Polinom deneyini `klima` özelliği ile tekrarlayın (X = klima). En iyi derece ve test RMSE'si değişti mi? Neden?

In [ ]:
# Alıştırma 3

**Alıştırma 4.** `KFold(5, shuffle=True)` ile aynı doğrusal modeli değerlendirin; `TimeSeriesSplit` sonucuyla karşılaştırın. Hangisi daha iyimser, neden?

In [ ]:
# Alıştırma 4

**Alıştırma 5.** Test kümesini ilk 3 gün / son 3 gün olarak ikiye bölüp saatlik profil tabanının RMSE'sini ayrı ayrı hesaplayın. Fark var mı? Son günlerin sıcaklığına bakarak yorumlayın.

In [ ]:
# Alıştırma 5